### Import Libraries and data

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [2]:
def reduce_mem_usage(df):

    start_mem = df.memory_usage().sum() 
    print('Memory usage of dataframe is {:.2f} MB'.format(start_mem))
    
    for col in df.columns:
        col_type = df[col].dtype
        
        if col_type != object:
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)  
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)
        else:
            df[col] = df[col].astype('category')

    end_mem = df.memory_usage().sum() 
    print('Memory usage after optimization is: {:.2f} MB'.format(end_mem))
    print('Decreased by {:.1f}%'.format(100 * (start_mem - end_mem) / start_mem))
    return df

In [3]:
sample_feature = reduce_mem_usage(pd.read_csv('data_for_tree.csv'))

Memory usage of dataframe is 63691972.00 MB
Memory usage after optimization is: 17117446.00 MB
Decreased by 73.1%


In [4]:
continuous_feature_names = [x for x in sample_feature.columns if x not in ['price','brand','model','brand']]

### Linear Regression & 5-folds CV & Simulation of reality

- Simple modeling

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

In [7]:
df = sample_feature.copy()

df = df.replace('-', np.nan)
df['price'] = pd.to_numeric(df['price'], errors='coerce')

if 'continuous_feature_names' in globals():
    num_cols = [c for c in continuous_feature_names if c in df.columns]
else:
    num_cols = df.select_dtypes(include='number').columns.drop('price', errors='ignore').tolist()

X_raw = df[num_cols].copy()
y_raw = df['price'].copy()

# inf -> NaN
X_raw.replace([np.inf, -np.inf], np.nan, inplace=True)

print("Original Shape:", df.shape, "X_raw:", X_raw.shape, "y_raw not None:", y_raw.notna().sum())

min_non_na = max(1, int(0.7 * len(num_cols))) if len(num_cols) > 0 else 1
keep = y_raw.notna() & (X_raw.notna().sum(axis=1) >= min_non_na)

if keep.sum() == 0:
    keep = y_raw.notna() & (X_raw.notna().sum(axis=1) >= 1)

X = X_raw.loc[keep]
y = y_raw.loc[keep]

print("After selected:", len(X))

too_empty = X.isna().mean() > 0.98
X = X.loc[:, ~too_empty]

test_size = 0.25 if len(X) >= 20 else 0.2
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=test_size, random_state=42
)

print("X_train:", X_train.shape, "X_test:", X_test.shape)

model = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('lr', LinearRegression())
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print("R2:", r2_score(y_test, y_pred))
print("MAE:", mean_absolute_error(y_test, y_pred))

Original Shape: (199037, 40) X_raw: (199037, 37) y_raw not None: 149037
After selected: 149037
X_train: (111777, 36) X_test: (37260, 36)
R2: 0.7333661887428851
MAE: 2336.7544170247434
